# 03 · Single-event benchmarks: METABRIC and SUPPORT (paper §4.1)

Re-runs the SurvTRACE-protocol benchmark **after the fixes** (see README):
train/val were swapped (models trained on ~10% of the data), Cox PH ignored every
categorical feature, and model selection used a known-broken C-index. Every earlier
METABRIC/SUPPORT number is therefore superseded.

* Stage A tunes the shared transformer backbone on validation C_td (SurvTRACE's grid).
* Stage B: Cox PH, DeepHit, DSM, MENSA (as published) and our model with the loss
  ablation + MMV λ_v, 5 seeds × 5 different splits.

## Options

In [ ]:
NAME = "03_single_event_benchmarks"

DATASETS = ["metabric", "support"]
SEEDS    = [0, 1, 2, 3, 4]          # seed s = init seed s AND split s

# Stage A: backbone tuning on the validation C_td (SurvTRACE's grid), recipe L_PCH, split 0.
TUNE = True
TUNE_GRID = {
    "transformer_num_hidden_layers": [2, 3, 4],
    "transformer_hidden_size": [8, 16],
    "transformer_intermediate_size": [32, 64],
    "learning_rate": [1e-3, 1e-4],
}

BASELINES = ["coxph", "deephit_paper", "dsm_paper", "mensa_paper"]
# (recipe, lambda_v); lambda_v=None for recipes without MMV. The earlier choice
# (lambda_v = 1, scale balancing) was made on runs with the train/val swap, so it is re-checked.
OUR_RECIPES = [
    ("nllpch", None),
    ("nllpch_sample_ranking", None),
    ("nllpch_mmv", 0.01),
    ("nllpch_mmv_rank", 0.01),
    ("nllpch_mmv_rank", 0.1),
    ("nllpch_mmv_rank", 1.0),
]

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/version_1/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Datasets

In [ ]:
GROUP = {"metabric": "survtrace_metabric", "support": "survtrace_support"}
HUB   = {"metabric": "metabric_numeric", "support": "support_numeric"}
if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    TUNE_GRID = {"transformer_num_hidden_layers": [2, 3]}
for ds in DATASETS:
    r.prepare(f"{GROUP[ds]}/survival", [], label=ds)

## Stage A · tune the backbone (validation C_td only — the test split is never looked at)

In [ ]:
import itertools
import pandas as pd
TUNED = {ds: [] for ds in DATASETS}
if TUNE:
    keys = list(TUNE_GRID)
    grid = [dict(zip(keys, v)) for v in itertools.product(*TUNE_GRID.values())]
    tune_runs = []
    for ds in DATASETS:
        for i, cfg in enumerate(grid):
            ov = [f"{k}={v}" for k, v in cfg.items()] + ["seed=0", "split_seed=0", "tasks/losses=nllpch"]
            tune_runs.append(Run(f"tune__{ds}__{i:02d}", "finetune", f"{GROUP[ds]}/survival", ov, HUB[ds],
                                 {"dataset": ds, "model": "tune", "grid_id": i, **cfg}))
    r.run_many(tune_runs, time_budget_min=TIME_BUDGET_MIN)
    val = r.collect(split="validation")
    val = val[val["model"] == "tune"]
    for ds in DATASETS:
        sub = val[val["dataset"] == ds].sort_values("ctd_weighted_avg", ascending=False)
        if len(sub):
            best = sub.iloc[0]
            TUNED[ds] = [f"{k}={best[k]:g}" if isinstance(best[k], float) else f"{k}={int(best[k])}" for k in keys]
            display(sub[keys + ["ctd_weighted_avg"]].head(5).rename(columns={"ctd_weighted_avg": "val C_td"}))
    (r.results_dir / "tuned_backbone.json").write_text(json.dumps(TUNED, indent=2))
print("backbone overrides:", TUNED)

## Stage B · benchmark (all neural models share the tuned backbone)

In [ ]:
runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for ds in DATASETS:
        g, hub, bb, base = GROUP[ds], HUB[ds], TUNED[ds], {"dataset": ds, "seed": s}
        for m in BASELINES:
            runs.append(Run(f"{ds}__{m}__s{s}", "coxph" if m == "coxph" else "finetune",
                            f"{g}/{m}", so + ([] if m == "coxph" else bb), hub, base | {"model": m}))
        for rec, lv in OUR_RECIPES:
            ov = [f"tasks/losses={rec}"] + ([f"mmv_variance_weight={lv}"] if lv is not None else [])
            tag = f"{ds}__ours_{rec}" + (f"_lv{lv:g}" if lv is not None else "") + f"__s{s}"
            runs.append(Run(tag, "finetune", f"{g}/survival", so + bb + ov, hub,
                            base | {"model": "ours", "recipe": rec, "lambda_v": lv}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results
C_td at the 25/50/75% horizons is directly comparable with SurvTRACE Table 2. Error bars now include split variation (different split per seed), like SurvTRACE's.

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

In [ ]:
# Horizon table in SurvTRACE's layout, with the published rows next to ours.
# Published values as recorded in summery.md from SurvTRACE (Wang & Sun 2022) Table 2 -
# check them against the paper before citing.
PUBLISHED = {
    ("metabric", "Cox PH"): (0.628, 0.627, 0.632), ("metabric", "DeepHit"): (0.712, 0.657, 0.603),
    ("metabric", "DSM"): (0.707, 0.663, 0.636), ("metabric", "PC-Hazard"): (0.713, 0.680, 0.644),
    ("support", "Cox PH"): (0.549, 0.564, 0.586), ("support", "DeepHit"): (0.650, 0.602, 0.574),
    ("support", "DSM"): (0.640, 0.609, 0.596), ("support", "PC-Hazard"): (0.652, 0.620, 0.607),
}
hcols = ["ctd_0th_event_0.25", "ctd_0th_event_0.5", "ctd_0th_event_0.75"]
res = df[df["model"] != "tune"]
if len(res):
    g = res.groupby(["dataset", "label"])[hcols].agg(["mean", "std"])
    tab = pd.DataFrame({h.split("_")[-1]: [f"{m:.3f} ± {s:.3f}" for m, s in zip(g[(h, "mean")], g[(h, "std")].fillna(0))]
                        for h in hcols}, index=g.index)
    pub = pd.DataFrame([{"dataset": d, "label": f"{n} (published)", "0.25": f"{a:.3f}", "0.5": f"{b:.3f}", "0.75": f"{c:.3f}"}
                        for (d, n), (a, b, c) in PUBLISHED.items() if d in DATASETS]).set_index(["dataset", "label"])
    horizon = pd.concat([tab, pub]).sort_index()
    horizon.to_csv(r.results_dir / "report" / "horizons_vs_published.csv")
    display(horizon)

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)